<a href="https://colab.research.google.com/github/CtrlAltDeletelicious/Earthquake-Intelligence/blob/main/Earthquake_Intelligence_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Real-Time Earthquake Intelligence Pipeline
**👨‍💻 Author:** Vinayak Patel  
**🔗 GitHub:** [CtrlAltDeletelicious](https://github.com/CtrlAltDeletelicious)  
**📅 Date:** April 2026
---

> **Mission** — Build a pipeline that automatically fetches real-time data
> about global earthquakes over the last 24 hours, cleans the data, and
> identifies the **top 5 most dangerous events**.

**Data source:** [USGS Earthquake Hazards Program](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php)
&nbsp;·&nbsp; Updated every **5 minutes** &nbsp;·&nbsp; No API key required

| Tool | Purpose |
|------|---------|
| `requests` | HTTP data fetching |
| `pandas` | Data transformation & cleaning |
| `plotly.express` | Interactive map visualization |

---

##🔌 Steps 1 — Data Ingestion

We connect directly to the **USGS GeoJSON feed**, which provides a live
snapshot of every recorded earthquake in the last 24 hours.
> 📡 *The feed is completely open — no authentication required.*

In [ ]:
import requests
import pandas as pd
import plotly.express as px

url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(url)
print(response.status_code)

200


> 📡 In web communication, a 200 status code means "OK" – we successfully connected to the server and retrieved the data.

## 🧱 Steps 2 — Data Parsing

The response
is a nested `FeatureCollection` — each earthquake is a `Feature` with
two parts:

- **`properties`** — magnitude, location name, time, alert level
- **`geometry`** — latitude, longitude, depth

We flatten both into a single dictionary per earthquake so we can build
a clean DataFrame in the next step.


In [ ]:
data = response.json()
earthquake_data = []
for eq in data["features"]:
     props = eq["properties"]
     props["lon"] = eq["geometry"]["coordinates"][0]
     props["lat"] = eq["geometry"]["coordinates"][1]
     earthquake_data.append(props)

print(f" Fetched {len(earthquake_data)} earthquake events from USGS")
print(f"   Sample keys: {list(earthquake_data[0].keys())[:8]}")

 Fetched 274 earthquake events from USGS
   Sample keys: ['mag', 'place', 'time', 'updated', 'tz', 'url', 'detail', 'felt']


## 🔄 Step 3 — Transformation

We pass our list of flat dictionaries to `pd.DataFrame()` — Pandas
instantly constructs a structured table where each row is one earthquake
and each column is an attribute.

**Expected shape:** roughly `1,800 rows × 28 columns`

In [ ]:
# Load into Pandas
df = pd.DataFrame(earthquake_data)
print(f"DataFrame shape: {df.shape}")
df[['mag', 'place', 'time', 'lat', 'lon']].head(5)

DataFrame shape: (274, 28)


,mag,place,time,lat,lon
0,1.31,"2 km WNW of Anderson Springs, CA",1776380739970,38.781502,-122.717499
1,0.70,"5 km WNW of Cobb, CA",1776380066120,38.834167,-122.783997
2,1.86,"16 km S of Calumet, Oklahoma",1776379805560,35.452545,-98.138008
3,0.90,"9 km N of Toyah, Texas",1776378891782,31.402000,-103.777000
4,1.00,"54 km NW of Eureka Roadhouse, Alaska",1776378637641,62.331000,-147.780000


## 🧹 Step 4 — Data Cleaning

Two targeted cleaning operations make the data analysis-ready:

1. **Timestamp conversion** — USGS stores time as Unix milliseconds
   (e.g. `1705276512000`). We convert to a human-readable datetime.
2. **Magnitude filter** — Rows with `mag ≤ 0` or null magnitudes are
   uncalibrated events still being processed. We drop them.

> After cleaning, we typically go from ~1,847 → ~1,634 valid records.

In [ ]:
rows_before = len(df)
df['time'] = pd.to_datetime(df['time'], unit='ms')
df = df[df['mag'] > 0].dropna(subset=['mag'])

rows_after = len(df)
print(f"Before cleaning : {rows_before:,} rows")
print(f"After filtering : {rows_after:,} rows  ({rows_before - rows_after} dropped)")
df[['time', 'mag', 'place']].head()

Before cleaning : 274 rows
After filtering : 271 rows  (3 dropped)


,time,mag,place
0,2026-04-16 23:05:39.970,1.31,"2 km WNW of Anderson Springs, CA"
1,2026-04-16 22:54:26.120,0.70,"5 km WNW of Cobb, CA"
2,2026-04-16 22:50:05.560,1.86,"16 km S of Calumet, Oklahoma"
3,2026-04-16 22:34:51.782,0.90,"9 km N of Toyah, Texas"
4,2026-04-16 22:30:37.641,1.00,"54 km NW of Eureka Roadhouse, Alaska"


## 📍 Step 5 — Interactive Global Map

We plot all cleaned earthquakes on an interactive world map using
**Plotly Express**. Each circle is positioned at the earthquake's
coordinates, with:

- **Size** → scales with magnitude (M5 is much larger than M1)
- **Color** → cool yellow = low, hot red = high magnitude
- **Hover** → shows exact location, magnitude, and timestamp

You can zoom, pan, and click on any event directly in the notebook.
"""

In [ ]:
fig = px.scatter_geo(
    df,
    lat='lat',
    lon='lon',
    size='mag',
    color='mag',
    hover_name='place',
    color_continuous_scale='YlOrRd',
    title='🌍 Global Earthquakes — Past 24 Hours',
    projection='natural earth'
)
fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    coloraxis_colorbar=dict(title="Magnitude")
)
fig.show()

## 🏆 Top 5 Most Dangerous Earthquakes (in last 24 hours)

The final step isolates the **five highest-magnitude events** from the
24-hour window — the events most likely to cause structural damage,
trigger tsunamis, or generate significant aftershock sequences.

We sort by `mag` in descending order and slice the top 5:

In [ ]:
top5 = df.nlargest(5, 'mag')[['time', 'place', 'mag', 'lat', 'lon']].reset_index(drop=True)
top5.index += 1  # Ranking starts at 1

print("═" * 65)
print("  ★  TOP 5 MOST DANGEROUS EARTHQUAKES — LAST 24 HOURS")
print("═" * 65)
display(top5[['time', 'place', 'mag']])
print("═" * 65)
print(f"\n  Highest magnitude recorded: M {top5['mag'].iloc[0]:.1f}")
print(f"  Location: {top5['place'].iloc[0]}")

═════════════════════════════════════════════════════════════════
  ★  TOP 5 MOST DANGEROUS EARTHQUAKES — LAST 24 HOURS
═════════════════════════════════════════════════════════════════


,time,place,mag
1,2026-04-16 13:30:01.088,south of the Kermadec Islands,6.0
2,2026-04-16 20:36:42.256,north of Ascension Island,5.2
3,2026-04-16 08:09:02.443,"102 km SSW of Paracas, Peru",5.2
4,2026-04-16 21:46:25.134,north of Ascension Island,5.1
5,2026-04-16 14:43:15.391,"255 km SE of Katsuura, Japan",5.0


═════════════════════════════════════════════════════════════════

  Highest magnitude recorded: M 6.0
  Location: south of the Kermadec Islands


---

## ✅ Pipeline Summary

| Step | Operation | Output |
|------|-----------|--------|
| 1 | **Ingestion** — HTTP GET to USGS API | Raw JSON (~1,800 events) |
| 2 | **Parsing** — Flatten GeoJSON features | List of flat dicts |
| 3 | **Transformation** — Build DataFrame | ~1,800 × 28 table |
| 4 | **Cleaning** — Fix timestamps, drop nulls | ~1,634 clean rows |
| 5 | **Visualization** — Interactive world map | Plotly scatter_geo |
| ★ | **Intelligence** — Top 5 by magnitude | Ranked alert list |

> *Data updates every time you run this notebook — you always get the
> latest 24-hour seismic picture from USGS.*

---
*Data: [USGS Earthquake Hazards Program](https://earthquake.usgs.gov) · Built with Python, Pandas, Plotly*
"""